# Planning Pattern Extension - Planner Plus ReAct Executor


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f643b6891-84f6-4672-aa1f-4724c5ad2d12_716x526-3.gif" alt="Planning pattern" width="500"/>

This extension notebook is closer to Figure 1 from the Lab 4 instructions. Instead of manually injecting a later observation, we let one agent plan the next evidence-gathering step and let a `ReactAgent` execute that step with real notebook tools.

Keep the two roles separate as you work:
- `PlanningAgent` decides what should happen next.
- `ReactAgent` gathers evidence with tools and returns an observation the planner can use.

This notebook is an advanced follow-up to `03_lab_notebook.ipynb`, not a replacement for it.


## Lab Question and Response Format

Use the staged artifacts to answer this advanced planning question:

**What timeline best explains the missing-phone interval, what missing dependency does the first executor pass expose, and how should the explanation change after network-related evidence is gathered?**

In this notebook, the planner-control helper returns one of two short formats:
- `NEXT_TASK: ...` means the planner wants the executor to gather one more piece of evidence.
- `FINISHED: ...` means the planner thinks the evidence is sufficient and the workflow can stop.

By the end of the notebook, the final report should be organized with these sections:
1. Planner step log
2. Direct-evidence findings
3. Network evidence findings
4. Final timeline and evidence-bounded conclusion

Note for students: `search_network_evidence(...)` only searches the local `data/` folder in this lab. It does not search the web.


### Step 1: Set Up the Notebook

Run this setup cell first. It loads the Lab 4 configuration from `.env`, checks that you opened the notebook from the correct folder, and prepares the client, imports, and data path.

To run a cell in Jupyter, click the cell and press `Shift+Enter`.


In [ ]:
import csv
import json
import os
import string
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

from agentic_patterns.planning_pattern.planning_agent import PlanningAgent
from agentic_patterns.react_pattern.react_agent import ReactAgent
from agentic_patterns.tool_pattern.tool import tool

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Data files:', sorted(path.name for path in data_dir.iterdir() if path.is_file()))


### Step 2: Define the Executor Tools

These tools are the evidence menu for the executor.

The first executor round will only receive the direct-evidence tools:
- `list_artifact_files()`
- `get_incident_window()`
- `get_unlock_events()`
- `get_call_log()`
- `get_whatsapp_events()`

The second executor round adds one more tool:
- `search_network_evidence(query)`

For teaching purposes, the first visible artifact list intentionally omits `network_status.csv`. The initial planner and executor should notice a missing connectivity dependency first, then use the network-search step to surface the withheld network artifact.

Each tool returns a short structured summary so the notebook stays readable.


In [ ]:
# Helper function: read one CSV evidence file and return its rows.
def read_csv(filename: str) -> list[dict]:
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


# Helper function: return one JSON string with indentation for readability.
def as_pretty_json(payload: dict | list) -> str:
    return json.dumps(payload, indent=2)


# This list simulates the first visible evidence bundle for the planner and the first executor round.
# `network_status.csv` is intentionally withheld here so the later network-search step can discover it.
initial_visible_artifact_files = [
    'artifact_manifest.json',
    'call_log.csv',
    'chain_of_custody.csv',
    'unlock_events.csv',
    'whatsapp_events.csv',
]


# Tool 1: show which evidence files are visible in the first pass of the case.
@tool
def list_artifact_files() -> str:
    """Return the initially visible artifact files for the first Lab 4 evidence pass."""
    return as_pretty_json({'artifact_files': initial_visible_artifact_files})


# Tool 2: return the main incident window from the case manifest.
@tool
def get_incident_window() -> str:
    """Return the UTC start and end timestamps for the missing-phone interval."""
    manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))
    incident_window = manifest['incident_window_utc']
    return as_pretty_json(
        {
            'start': incident_window['start'],
            'end': incident_window['end'],
            'analysis_timezone': manifest['analysis_timezone'],
        }
    )


# Tool 3: return the unlock and lock events that bound device access in the case.
@tool
def get_unlock_events() -> str:
    """Return the recorded unlock and lock events for the device."""
    rows = read_csv('unlock_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 4: return the phone call log summary.
@tool
def get_call_log() -> str:
    """Return the recorded phone call events for the incident period."""
    rows = read_csv('call_log.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 5: return the WhatsApp activity summary.
@tool
def get_whatsapp_events() -> str:
    """Return the WhatsApp activity relevant to the incident period."""
    rows = read_csv('whatsapp_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 6: search the local evidence files for network-related clues.
@tool
def search_network_evidence(query: str) -> str:
    """Search the local Lab 4 data folder for network-related evidence that matches the query."""
    query_terms = {
        token.strip(string.punctuation).lower()
        for token in query.split()
        if token.strip(string.punctuation)
    }
    if not query_terms:
        query_terms = {'network'}

    matches = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        file_matches = []
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            lower_line = line.lower()
            if any(term in lower_line for term in query_terms):
                file_matches.append({'line_number': line_number, 'text': line})
            if len(file_matches) >= 4:
                break
        if file_matches:
            matches.append({'file': path.name, 'matches': file_matches})

    if not matches:
        fallback_lines = (data_dir / 'network_status.csv').read_text(encoding='utf-8').splitlines()[:4]
        return as_pretty_json(
            {
                'query': query,
                'matches': [],
                'note': 'No exact keyword match found. Showing the start of network_status.csv as a fallback.',
                'fallback_excerpt': fallback_lines,
            }
        )

    return as_pretty_json({'query': query, 'matches': matches})


# The first executor round gets only the direct evidence tools.
direct_tools = [
    list_artifact_files,
    get_incident_window,
    get_unlock_events,
    get_call_log,
    get_whatsapp_events,
]

# The second executor round adds the network-search tool for replanning.
expanded_tools = direct_tools + [search_network_evidence]

tool_schema_preview = {
    tool.name: json.loads(tool.fn_signature)
    for tool in expanded_tools
}

tool_schema_preview


### Step 3: Let `PlanningAgent` Build the Initial Investigation Plan

The planner starts from the case question plus a high-level artifact list. This is intentional: the planner decides what evidence should be gathered next, but it has not inspected the detailed rows yet.

That initial artifact list also excludes `network_status.csv`, so the first plan should surface a missing connectivity dependency instead of solving it immediately.


In [ ]:
planning_agent = PlanningAgent(client=client, model=MODEL)

case_question = (
    'What timeline best explains the missing-phone interval, what missing dependency does the first executor pass expose, '
    'and how should the explanation change after network-related evidence is gathered?'
)

artifact_menu = {
    'artifact_files': initial_visible_artifact_files,
    'note': 'The planner should choose what the executor needs to inspect next from this initial visible bundle.',
}

planner_case_context = (
    f'{case_question}\n\n'
    f'High-level artifact list:\n{json.dumps(artifact_menu, indent=2)}'
)

initial_plan = planning_agent.build_initial_plan(planner_case_context)
display(Markdown('### Initial Planner Output\n\n' + initial_plan))


### Step 4: Ask the Planner for the Next Single Executor Task

This helper asks the planner for exactly one next step. The answer should be short enough for the executor to act on directly.

In the first round, the planner only sees the direct-evidence tool list, so it should not ask for network-search evidence yet.


In [ ]:
PLANNER_CONTROLLER_SYSTEM_PROMPT = """
You coordinate a planning workflow for a digital forensics case.

You are not the executor. Your job is to choose the next single evidence-gathering task for a ReactAgent.

Return exactly one of these two formats:
NEXT_TASK: <one concrete task sentence>
FINISHED: <brief reason and final reporting instruction>

Rules:
- Ask for only one task at a time.
- The task must be solvable with the listed executor tools.
- Prefer direct evidence before search-based evidence.
- If current observations expose a missing dependency about connectivity, delivery, or online/offline status, ask next for network-related evidence.
- Keep the task short and specific.
""".strip()


# Ask the planner for the next task using the current plan and current observations.
def ask_planner_for_next_task(
    case_question: str,
    current_plan: str,
    observations: str,
    tool_names: list[str],
) -> str:
    observation_text = observations if observations else 'No executor observations yet.'
    prompt = (
        f'Case question:\n{case_question}\n\n'
        f'Current plan:\n{current_plan}\n\n'
        f'Observations so far:\n{observation_text}\n\n'
        f'Available executor tools:\n- ' + '\n- '.join(tool_names)
    )
    return client.chat.completions.create(
        messages=[
            {'role': 'system', 'content': PLANNER_CONTROLLER_SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        model=MODEL,
    ).choices[0].message.content.strip()


# Pull the task text out of a planner response such as "NEXT_TASK: ...".
def extract_next_task(planner_output: str) -> str | None:
    if 'NEXT_TASK:' in planner_output:
        return planner_output.split('NEXT_TASK:', 1)[1].strip()
    if planner_output.startswith('FINISHED:'):
        return None
    return planner_output.strip()


planner_decision_1 = ask_planner_for_next_task(
    case_question=case_question,
    current_plan=initial_plan,
    observations='',
    tool_names=[tool.name for tool in direct_tools],
)
display(Markdown('### Planner Decision for Executor Round 1\n\n' + planner_decision_1))

next_task_1 = extract_next_task(planner_decision_1)
print('Round 1 task:', next_task_1)


### Step 5: Run `ReactAgent` on the Direct-Evidence Task

This executor round can only inspect the direct artifacts. The goal is to surface the first timeline explanation and identify what dependency is still missing before a stronger conclusion can be made.


In [ ]:
direct_executor = ReactAgent(tools=direct_tools, client=client, model=MODEL)

executor_prompt_1 = f"""
Carry out this planner-assigned task using the available tools:

{next_task_1}

Requirements:
- Collect the direct evidence needed to describe the missing-phone interval.
- Mention the key timestamps you find.
- If the case still depends on network availability or delivery status, say that this is a missing dependency.
- Return a short response with these labels:
  Direct evidence gathered:
  Missing dependency or uncertainty:
""".strip()

executor_result_1 = direct_executor.run(user_msg=executor_prompt_1)
display(Markdown('### Executor Round 1 Final Response\n\n' + executor_result_1))


### Step 6: Feed the First Executor Result Back to the Planner

Now the planner sees a real observation from the executor rather than a manually injected note. It should revise the plan and choose the next task.

In this round, the available tool list expands to include `search_network_evidence(...)`, which can surface the withheld network artifact and the connectivity timestamps that matter for delivery timing.


In [ ]:
revised_plan_1 = planning_agent.revise_plan(
    user_msg=planner_case_context,
    current_plan=initial_plan,
    observations=f'Executor round 1 result:\n{executor_result_1}',
)
display(Markdown('### Revised Planner Output After Round 1\n\n' + revised_plan_1))

planner_decision_2 = ask_planner_for_next_task(
    case_question=case_question,
    current_plan=revised_plan_1,
    observations=executor_result_1,
    tool_names=[tool.name for tool in expanded_tools],
)
display(Markdown('### Planner Decision for Executor Round 2\n\n' + planner_decision_2))

next_task_2 = extract_next_task(planner_decision_2)
print('Round 2 task:', next_task_2)


### Step 7: Run `ReactAgent` Again with Network Search Added

This second executor round can use the network-search tool. The idea is to let the planner ask for the missing evidence instead of having the notebook reveal it automatically.


In [ ]:
expanded_executor = ReactAgent(tools=expanded_tools, client=client, model=MODEL)

executor_prompt_2 = f"""
Carry out this planner-assigned task using the available tools:

{next_task_2}

Requirements:
- If the task is about connectivity, use the tool that can search for network-related evidence in the local Lab 4 data folder.
- Report the important file name, timestamps, and online/offline status you find.
- Return a short response with these labels:
  Network evidence gathered:
  Impact on the timeline:
""".strip()

executor_result_2 = expanded_executor.run(user_msg=executor_prompt_2)
display(Markdown('### Executor Round 2 Final Response\n\n' + executor_result_2))


### Step 8: Let the Planner Write the Final Report

The planner now has the initial plan, the revised plan, and two executor observations. This is the point where it should produce the bounded final explanation instead of continuing to collect evidence.


In [ ]:
combined_observations = (
    'Executor round 1 result:\n'
    f'{executor_result_1}\n\n'
    'Executor round 2 result:\n'
    f'{executor_result_2}'
)

revised_plan_2 = planning_agent.revise_plan(
    user_msg=planner_case_context,
    current_plan=revised_plan_1,
    observations=combined_observations,
)
display(Markdown('### Planner Update After Network Evidence\n\n' + revised_plan_2))

final_report_prompt = (
    'Use the case question, the current investigation plan, and the executor observations below to write the final forensic report.\n\n'
    'Return a report with:\n'
    '1. Planner step log\n'
    '2. Direct-evidence findings\n'
    '3. Network evidence findings\n'
    '4. Final timeline and evidence-bounded conclusion\n\n'
    f'Case question:\n{case_question}\n\n'
    f'Current plan:\n{revised_plan_2}\n\n'
    f'Executor observations:\n{combined_observations}'
)

final_report = planning_agent._complete(final_report_prompt)
display(Markdown('### Planner Final Report\n\n' + final_report))


## Simplified Planning vs. Planner-plus-ReAct Planning

`03_lab_notebook.ipynb` kept the planning pattern simple by withholding the network artifact and then revealing it later. This extension notebook is closer to the general planning figure: the planner decides what to do next, the executor gathers evidence with tools, and the planner revises the path using those observations.

Both notebooks teach the same caution: students should not overclaim message delivery when the evidence only supports activity plus later connectivity context.
